# 人体动作识别 — 1D-CNN (基于手工提取特征)

使用 UCI HAR Dataset 的 561 维手工提取特征，通过一维卷积神经网络识别 6 种日常动作。

## 1. 导入依赖

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Conv1D, Dense, Dropout, MaxPooling1D,
    BatchNormalization, GlobalAveragePooling1D,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

## 2. 超参数配置

In [2]:
N_CLASSES  = 6
BATCH_SIZE = 64
MAX_EPOCHS = 100

## 3. 数据加载

读取 UCI HAR Dataset 中已提取好的 561 维特征，并将其 reshape 为 (561, 1) 以适配 Conv1D 输入。

- Conv1D 要求输入形状为 `(时间步, 特征)`
- 此处将 561 个特征视为 561 个时间步，每个时间步 1 个特征通道

In [3]:
def load_data():
    x_train = pd.read_csv('UCI HAR Dataset/train/X_train.txt', header=None, sep='\\s+').values
    y_train = pd.read_csv('UCI HAR Dataset/train/y_train.txt', header=None, sep='\\s+').values
    x_test  = pd.read_csv('UCI HAR Dataset/test/X_test.txt',   header=None, sep='\\s+').values
    y_test  = pd.read_csv('UCI HAR Dataset/test/y_test.txt',   header=None, sep='\\s+').values

    # 从原训练集中划分验证集（20%）
    n_train = x_train.shape[0]
    indices = np.random.permutation(n_train)
    val_size = int(n_train * 0.2)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    x_val = x_train[val_indices]
    y_val = y_train[val_indices]
    x_train = x_train[train_indices]
    y_train = y_train[train_indices]

    # Reshape: (N, 561) -> (N, 561, 1)，适配 Conv1D 输入
    x_train = x_train[..., np.newaxis]
    x_val   = x_val[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]

    # 标签处理: 1-6 转 0-5，再转 One-Hot
    y_train = to_categorical(y_train - 1)
    y_val   = to_categorical(y_val - 1)
    y_test  = to_categorical(y_test - 1)

    print(f'训练集 {x_train.shape}, 验证集 {x_val.shape}, 测试集 {x_test.shape}')
    return x_train, y_train, x_val, y_val, x_test, y_test

x_train, y_train, x_val, y_val, x_test, y_test = load_data()

训练集 (5882, 561, 1), 验证集 (1470, 561, 1), 测试集 (2947, 561, 1)


## 4. 模型结构 — 1D-CNN

输入形状 (561, 1)，三层卷积 + GAP + 全连接：

```
Conv1D(64, 3) → BN → Conv1D(64, 3) → BN → MaxPool → Dropout
  → Conv1D(128, 3) → BN → MaxPool → Dropout → GAP
  → Dense(128) → BN → Dropout(0.5)
  → Dense(64) → BN → Dropout(0.5)
  → Softmax(6)
```

In [4]:
def build_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(64, 3, activation='relu', padding='same',
               kernel_regularizer=l2(1e-4))(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, activation='relu', padding='same',
               kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)
    x = Conv1D(128, 3, activation='relu', padding='same',
               kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.3)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(N_CLASSES, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(1e-3),
                  metrics=['accuracy'])
    return model

model = build_model(x_train.shape[1:])
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 561, 1)]          0         
                                                                 
 conv1d (Conv1D)             (None, 561, 64)           256       
                                                                 
 batch_normalization (BatchN  (None, 561, 64)          256       
 ormalization)                                                   
                                                                 
 conv1d_1 (Conv1D)           (None, 561, 64)           12352     
                                                                 
 batch_normalization_1 (Batc  (None, 561, 64)          256       
 hNormalization)                                                 
                                                                 
 max_pooling1d (MaxPooling1D  (None, 280, 64)          0     

## 5. 训练与评估

In [ ]:
model.fit(
    x_train, y_train,
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ],
    verbose=1,
)

_, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'测试集准确率: {test_acc * 100:.2f}%')

Epoch 1/100
92/92 [==============================] - 5s 9ms/step - loss: 1.6145 - accuracy: 0.3975 - val_loss: 1.8902 - val_accuracy: 0.3333 - lr: 0.0010
Epoch 2/100
92/92 [==============================] - 1s 7ms/step - loss: 1.1516 - accuracy: 0.5394 - val_loss: 2.3209 - val_accuracy: 0.1639 - lr: 0.0010
Epoch 3/100
92/92 [==============================] - 1s 7ms/step - loss: 0.9163 - accuracy: 0.6142 - val_loss: 4.4412 - val_accuracy: 0.1639 - lr: 0.0010
Epoch 4/100
92/92 [==============================] - 1s 9ms/step - loss: 0.7501 - accuracy: 0.6812 - val_loss: 7.2904 - val_accuracy: 0.1639 - lr: 0.0010
Epoch 5/100
92/92 [==============================] - 1s 10ms/step - loss: 0.6583 - accuracy: 0.7164 - val_loss: 9.0342 - val_accuracy: 0.1639 - lr: 0.0010
Epoch 6/100
92/92 [==============================] - 1s 11ms/step - loss: 0.6024 - accuracy: 0.7428 - val_loss: 9.1741 - val_accuracy: 0.1639 - lr: 0.0010
Epoch 7/100
92/92 [==============================] - 1s 11ms/step - loss: 

: 